# Comparison program

Canonical source for `compete_detect.py`. Export the complete fresh comparison runtime from the repository root:

```sh
python bench/comparison_loader.py NEW_DIRECTORY
```

Use the original command documented below with the exported program path. External runtimes and native defaults remain those specified by the program. This notebook migration changes source ownership, not model parameters or recorded predictions.


In [ ]:
"""Run the new competitors (pMTnet, TCRGP, TCR-BERT) on the held-out annotation set and score per-epitope
ROC + IMMREP AUC0.1, on the SAME pooled query set vdjmatch uses (pos = true==E, neg = all other queries).

All three are VDJdb-trained -> our held-out positives are likely in their training data: report with the
leakage caveat (per the user's decision: full naive re-run, leakage noted in text).

    .venv/bin/python bench/compete_detect.py pmtnet TRB
    .venv/bin/python bench/compete_detect.py tcrgp  TRB
    conda run -n cmp-tcrbert python bench/compete_detect.py tcrbert TRB
"""
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
import _bench                                                        # noqa: E402
import holdout_eval as HE                                            # noqa: E402
from holdout_features import _jmap                                   # noqa: E402
from metrics import auc01, roc_auc                                   # noqa: E402

CODE = Path.home() / "vcs/code"
RES = Path.home() / "vcs/manuscripts/2026-vdjmatch/benchmarks/results"
PRED = Path(__file__).resolve().parent / "predictions"
A2 = "A*02:01"   # pMTnet matches HLA by key.startswith(input); IMGT keys are like A*02:01:01:01
EPI = HE.EPI                                                         # {sh: peptide}
TCRGP_SHIPPED = {"NLV", "LLW", "GLC", "YLQ"}                         # ELA/LLL have no shipped model


def load_pool(locus):
    """pooled held-out queries: (cdr3, v, j, true_epi_aa_or_None), valid CDR3 only."""
    jm = _jmap(locus)
    out = []
    for c, v, true, ds in HE.query_set(locus):
        if _bench.valid_cdr3(pl.DataFrame({"cdr3": [c]})).height:
            out.append((c, v, jm.get(c, ""), true))
    return out


def write_results(tool, locus, pool, scores):
    """scores: {epitope_aa: list aligned to pool}. Append per-epitope ROC/AUC0.1 to competitors_holdout.tsv."""
    rows = []
    for sh, e in EPI.items():
        sc = scores.get(e)
        if sc is None:
            continue
        y = [int(t == e) for _, _, _, t in pool]
        pairs = list(zip(y, sc))
        rows.append(dict(tool=tool, locus=locus, epitope=sh, n_pos=sum(y),
                         roc=round(roc_auc(pairs), 4), auc01=round(auc01(pairs), 4)))
    df = pl.DataFrame(rows)
    f = RES / "competitors_holdout.tsv"
    if f.exists():
        old = pl.read_csv(f, separator="\t").filter(~((pl.col("tool") == tool) & (pl.col("locus") == locus)))
        df = pl.concat([old, df], how="diagonal")
    df.write_csv(f, separator="\t")
    print(df.filter((pl.col("tool") == tool) & (pl.col("locus") == locus)))


# ---------------------------------------------------------------- pMTnet (TRB only) -------------------
def run_pmtnet(locus, pool):
    """Per-epitope, resumable: pMTnet is run once per epitope (~3 min each) and the prediction cached as
    pred_<epi>.csv, so a kill loses at most one epitope. Re-running skips finished epitopes."""
    assert locus == "TRB", "pMTnet is beta-only"
    work = PRED / "pmtnet"; work.mkdir(parents=True, exist_ok=True)
    repo = CODE / "pMTnet"
    scores = {}
    for sh, e in EPI.items():
        pred_f = work / f"pred_{sh}.csv"
        if not pred_f.exists():
            inp = work / f"input_{sh}.csv"
            pl.DataFrame([(c, e, A2) for (c, _, _, _) in pool], schema=["CDR3", "Antigen", "HLA"],
                         orient="row").write_csv(inp)
            for attempt in range(8):                               # Rosetta-TF1 segfaults intermittently
                (work / "prediction.csv").unlink(missing_ok=True)
                subprocess.run(["conda", "run", "-n", "cmp-pmtnet", "python", "pMTnet.py", "-input", str(inp),
                                "-library", "library", "-output", str(work), "-output_log", str(work / "log.txt")],
                               cwd=repo, check=False)
                if (work / "prediction.csv").exists():
                    (work / "prediction.csv").rename(pred_f)
                    print(f"[pmtnet] {sh} done (attempt {attempt + 1}) -> {pred_f.name}", file=sys.stderr)
                    break
                print(f"[pmtnet] {sh} segfaulted, retry {attempt + 1}/8", file=sys.stderr)
            else:
                raise RuntimeError(f"pmtnet {sh} failed after 8 attempts (Rosetta-TF1 segfault)")
        pred = pl.read_csv(pred_f)
        rank = {c: r for c, r in zip(pred["CDR3"], pred["Rank"])}
        scores[e] = [1.0 - rank.get(c, 1.0) for (c, _, _, _) in pool]   # lower Rank = binder
    return scores


# ---------------------------------------------------------------- TCRGP (beta, shipped epitopes) ------
def run_tcrgp(locus, pool):
    assert locus == "TRB", "TCRGP shipped models are beta-only"
    import numpy as np
    repo = CODE / "TCRGP"
    work = PRED / "tcrgp"; work.mkdir(parents=True, exist_ok=True)
    inp = work / "input.csv"
    pl.DataFrame({"cdr3b": [c for c, _, _, _ in pool]}).write_csv(inp)
    scores = {}
    for sh in TCRGP_SHIPPED:
        e = EPI[sh]
        out = work / f"{sh}.npy"
        if not out.exists():                                       # gpflow1 tf_should_use exits non-zero
            subprocess.run(["conda", "run", "-n", "cmp-tcrgp", "python",   # even on success -> check=False
                            str(Path(__file__).resolve().parent / "_tcrgp_predict.py"), str(inp),
                            str(repo / "models" / "paper" / f"model_vdj_{e}_cdr3b"), str(out)], check=False)
        a = np.load(out)
        scores[e] = np.where(np.isnan(a), 0.0, a).tolist()         # length>23 CDR3s -> nan -> 0 (non-binder)
    return scores


# ---------------------------------------------------------------- TCR-BERT (embeddings + kNN density) -
def _bert_embed(seqs, tag):
    """embed via the cmp-tcrbert env (torch); cache npy under predictions/tcrbert/."""
    import numpy as np
    work = PRED / "tcrbert"; work.mkdir(parents=True, exist_ok=True)
    lst, out = work / f"{tag}.txt", work / f"{tag}.npy"
    lst.write_text("\n".join(seqs))
    subprocess.run(["conda", "run", "-n", "cmp-tcrbert", "python", str(Path(__file__).resolve().parent /
                    "_tcrbert_embed.py"), str(lst), str(out)], check=True)
    X = np.load(out)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)


def run_tcrbert(locus, pool):
    import numpy as np
    from benchmark import A02, release                              # noqa: E402
    from holdout_controls import _airr
    qcdr3 = [c for c, _, _, _ in pool]
    qX = _bert_embed(qcdr3, f"query_{locus}")
    rdf = _bench.valid_cdr3(release("vdjdb2026").filter(pl.col("mhc_a").str.contains(A02) & (pl.col("gene") == locus)))
    bgX = _bert_embed(_airr("human", locus, 4000)["cdr3"].to_list(), f"bg_{locus}")
    kbg = np.sort(qX @ bgX.T, axis=1)[:, -10:].mean(1)
    scores = {}
    for sh, e in EPI.items():
        ref = rdf.filter(pl.col("epitope") == e).unique("cdr3")["cdr3"].to_list()[:4000]
        if len(ref) < 20:
            continue
        rX = _bert_embed(ref, f"ref_{sh}_{locus}")
        S = qX @ rX.T                                              # density: top-10 cosine sim to E binders
        ridx = {c: j for j, c in enumerate(ref)}                  # EXCLUDE the exact self-match (vdjmatch's
        for i, c in enumerate(qcdr3):                             # leave-one-out protocol) so the memorizable
            j = ridx.get(c)                                       # epitopes (NLV/LLW/LLL/ELA) are fair
            if j is not None:
                S[i, j] = -1.0
        kpos = np.sort(S, axis=1)[:, -10:].mean(1)
        scores[e] = (kpos - kbg).tolist()
    return scores


if __name__ == "__main__":
    tool, locus = sys.argv[1], sys.argv[2] if len(sys.argv) > 2 else "TRB"
    pool = load_pool(locus)
    fn = {"pmtnet": run_pmtnet, "tcrgp": run_tcrgp, "tcrbert": run_tcrbert}[tool]
    write_results(tool, locus, pool, fn(locus, pool))
